In [11]:
from dotenv import load_dotenv
from agents import Agent, Runner, Trace
from pydantic import BaseModel
from agents.mcp import MCPServerStdio
import os

from IPython.display import display, Markdown, Image
import re

load_dotenv()

True

In [3]:
# System Prompt Templates:

agent1_instructions = ("You are a SQL Server analyst.\n"
            "Workflow:\n"
            "1) Use list_tables to find candidates.\n"
            "2) Use describe_table before writing SQL.\n"
            "3) Use execute_query for final SQL.\n"
            "4) Return concise results and include the SQL used.")

In [4]:
#MCP Params:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

sql_params={
        "command": "mssql-mcp-server",   # or "npx", "args": ["-y", "mssql-mcp-server"]
        "env": {
            "MSSQL_CONNECTION_STRING": os.getenv("MSSQL_CONNECTION_STRING"),
            
            # "MSSQL_HOST": os.getenv("MSSQL_HOST"),
            # "MSSQL_PORT": os.getenv("MSSQL_PORT", "1433"),
            # "MSSQL_DATABASE": os.getenv("MSSQL_DATABASE"),
            # "MSSQL_USER": os.getenv("MSSQL_USER"),
            # "MSSQL_PASSWORD": os.getenv("MSSQL_PASSWORD"),
            # "MSSQL_ENCRYPT": os.getenv("MSSQL_ENCRYPT", "false"),
            # "MSSQL_TRUST_SERVER_CERTIFICATE": os.getenv("MSSQL_TRUST_SERVER_CERTIFICATE", "true"),            
        }
}

chart_params = {
      "command": "npx", "args": ["-y", "@antv/mcp-server-chart"]
    }
  

In [5]:
fetch_server = MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60)

mssql_server = MCPServerStdio(params=sql_params, client_session_timeout_seconds=60)

chart_server = MCPServerStdio(params=chart_params, client_session_timeout_seconds=60)

In [6]:
async with mssql_server:
    print(await mssql_server.call_tool("test_connection", {}))

meta=None content=[TextContent(type='text', text='[\n  {\n    "version": "Microsoft SQL Server 2019 (RTM) - 15.0.2000.5 (X64) \\n\\tSep 24 2019 13:48:23 \\n\\tCopyright (C) 2019 Microsoft Corporation\\n\\tExpress Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\\n",\n    "server_name": "LAPTOP-CPI3JAKQ\\\\SQLEXPRESS",\n    "database_name": "Certainty_Durham2"\n  }\n]', annotations=None, meta=None)] structuredContent=None isError=False


In [7]:
async with mssql_server:
    tools = await mssql_server.list_tools()
    print([t.name for t in tools])

['list_connections', 'test_connection', 'list_databases', 'list_tables', 'describe_table', 'sample_data', 'execute_query', 'get_relationships', 'list_stored_procedures', 'describe_stored_procedure', 'list_views', 'describe_view', 'list_triggers', 'describe_trigger', 'list_functions', 'list_indexes', 'analyze_table_stats', 'find_missing_indexes', 'analyze_index_usage', 'analyze_database_size', 'list_constraints', 'analyze_check_constraints', 'list_user_defined_types', 'find_computed_columns', 'list_default_constraints', 'analyze_data_distribution', 'find_lookup_tables', 'analyze_null_patterns', 'detect_audit_columns', 'get_stored_procedure_definition', 'get_multiple_stored_procedure_definitions', 'get_all_stored_procedure_definitions', 'search_stored_procedures_by_content']


In [8]:
async with fetch_server:
    tools = await fetch_server.list_tools()
    print([t.name for t in tools])

['fetch']


In [9]:
async with chart_server:
    tools = await chart_server.list_tools()
    print([t.name for t in tools])

['generate_area_chart', 'generate_bar_chart', 'generate_boxplot_chart', 'generate_column_chart', 'generate_district_map', 'generate_dual_axes_chart', 'generate_fishbone_diagram', 'generate_flow_diagram', 'generate_funnel_chart', 'generate_histogram_chart', 'generate_line_chart', 'generate_liquid_chart', 'generate_mind_map', 'generate_network_graph', 'generate_organization_chart', 'generate_path_map', 'generate_pie_chart', 'generate_pin_map', 'generate_radar_chart', 'generate_sankey_chart', 'generate_scatter_chart', 'generate_treemap_chart', 'generate_venn_chart', 'generate_violin_chart', 'generate_waterfall_chart', 'generate_word_cloud_chart', 'generate_spreadsheet']


In [12]:
async with mssql_server, fetch_server, chart_server:
    agent = Agent(
            name="SQL Analyst",
            model="gpt-4o-mini",
            instructions=agent1_instructions,
            mcp_servers=[mssql_server, fetch_server, chart_server],
        )

    result = await Runner.run(
        agent,
        #"Show daily checklist completions for the last 30 days."
        #"Show daily checklist completions for the most recent 30 days that have data"
        "Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. Only generate one chart."
    )
    #print(result.final_output)

    display(Markdown(result.final_output))  # renders markdown tables/images if present

    # optional: if output contains markdown image link, render it explicitly
    m = re.search(r'!\[[^\]]*\]\(([^)]+)\)', result.final_output or "")
    if m:
        display(Image(url=m.group(1)))

Here are the results regarding daily completion counts for the 30 days ending on the latest CompletionDate:

### Latest Completion Date:
- **Latest Date:** 2025-02-06

### Daily Completion Counts:
| Completion Date | Completion Count |
|------------------|------------------|
| 2025-01-07       | 26               |
| 2025-01-08       | 15               |
| 2025-01-09       | 18               |
| 2025-01-10       | 15               |
| 2025-01-13       | 105              |
| 2025-01-14       | 26               |
| 2025-01-15       | 19               |
| 2025-01-16       | 23               |
| 2025-01-17       | 25               |
| 2025-01-19       | 2                |
| 2025-01-20       | 80               |
| 2025-01-21       | 36               |
| 2025-01-22       | 14               |
| 2025-01-23       | 15               |
| 2025-01-24       | 6                |
| 2025-01-26       | 1                |
| 2025-01-27       | 101              |
| 2025-01-28       | 18               |
| 2025-01-29       | 14               |
| 2025-01-30       | 18               |

### Chart
A line chart has been generated to visualize the daily completion counts over this period:

![Daily Completion Counts for Last 30 Days](https://mdn.alipayobjects.com/one_clip/afts/img/7M1cTb9CpWAAAAAATHAAAAgAoEACAQFr/original)

### SQL Queries Used
1. To find the latest date:
   ```sql
   SELECT MAX(CreationDate) AS LatestDate FROM dbo.CompletedChecklist;
   ```
2. To calculate daily counts:
   ```sql
   SELECT 
       CAST(CreationDate AS DATE) AS CompletionDate,
       COUNT(*) AS CompletionCount
   FROM dbo.CompletedChecklist
   WHERE CreationDate >= DATEADD(DAY, -30, '2025-02-06')
   GROUP BY CAST(CreationDate AS DATE)
   ORDER BY CompletionDate ASC;
   ``` 

If you need any further analysis or adjustments, let me know!